# Clinical Note Bias Assessment Tool (Ollama Version)

This notebook analyzes clinical notes for potentially biased or stigmatizing language using a local LLM via Ollama.

## Requirements

**Files needed:**
1. This notebook (`ollama_bias_assessment.ipynb`)
2. The prompt file (`bias_detection_prompt.py`) - must be in the same directory
3. Your input CSV file with a `note_text` column containing clinical notes

**Environment setup:**
1. Install Ollama from https://ollama.ai/

2. Pull a model (recommended models for this task):
   ```bash
   ollama pull llama3.1:8b        # Good balance of speed and quality
   ollama pull llama3.1:70b       # Higher quality, slower
   ollama pull mistral:7b         # Fast alternative
   ollama pull mixtral:8x7b       # High quality alternative
   ```

3. Make sure Ollama is running:
   ```bash
   ollama serve
   ```

4. Install required Python packages:
   ```bash
   pip install pandas requests
   ```

## Output

The tool generates a CSV file with two additional columns:
- **Possible_Biased_Terms**: Phrases that may reflect bias but are context-dependent
- **Likely_Biased_Terms**: Phrases that clearly reflect biased or stigmatizing language

## Configuration

Edit the settings below before running the assessment.

In [ ]:
from project_config import (
    BIAS_NOTES_CLEANED_CSV,
    BIAS_NOTES_OUTPUT_DIR,
    BIAS_NOTES_PROMPT_PATH,
)

# =============================================================================
# CONFIGURATION - Edit these settings before running
# =============================================================================

# Path to your input CSV file (must contain a 'note_text' column)
INPUT_CSV = BIAS_NOTES_CLEANED_CSV

# Output directory (where results will be saved)
OUTPUT_DIR = BIAS_NOTES_OUTPUT_DIR

# Output filename prefix (timestamp will be appended automatically)
OUTPUT_PREFIX = "bias_flagged_ollama"

# Path to the prompt file
PROMPT_PATH = BIAS_NOTES_PROMPT_PATH

# -----------------------------------------------------------------------------
# Ollama Settings
# -----------------------------------------------------------------------------

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.1:8b"

# -----------------------------------------------------------------------------
# Row Selection Settings
# -----------------------------------------------------------------------------

N_ROWS_TO_PROCESS = 100
RANDOM_SELECTION = True
RANDOM_SEED = None

CHUNK_CHAR_LIMIT = 2800
PARALLEL_CALLS = 2
TEMPERATURE = 0

if not INPUT_CSV:
    raise RuntimeError(
        "Set BIAS_NOTES_CLEANED_CSV in .env before running this notebook."
    )
if not OUTPUT_DIR:
    raise RuntimeError(
        "Set BIAS_NOTES_OUTPUT_DIR in .env before running this notebook."
    )
if not PROMPT_PATH:
    raise RuntimeError(
        "Set BIAS_NOTES_PROMPT_PATH in .env before running this notebook."
    )

## Run Assessment

Execute the cell below to run the bias assessment on your clinical notes.

In [ ]:
# =============================================================================
# BIAS ASSESSMENT ENGINE (OLLAMA) - Do not modify unless you know what you're doing
# =============================================================================

import os
import re
import json
import time
import random
from datetime import datetime
from typing import List, Dict
import concurrent.futures
from threading import Semaphore

import pandas as pd
import requests

# -----------------------------------------------------------------------------
# Ollama Client Setup
# -----------------------------------------------------------------------------


def check_ollama_connection():
    """Verify Ollama is running and the model is available."""
    try:
        # Check if Ollama is running
        resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        resp.raise_for_status()

        # Check if the specified model is available
        models = resp.json().get("models", [])
        model_names = [m.get("name", "") for m in models]

        # Check for exact match or partial match (model:tag format)
        model_found = any(
            OLLAMA_MODEL == name
            or OLLAMA_MODEL == name.split(":")[0]
            or name.startswith(OLLAMA_MODEL)
            for name in model_names
        )

        if not model_found:
            print(f"WARNING: Model '{OLLAMA_MODEL}' not found in Ollama.")
            print(f"Available models: {model_names}")
            print(f"\nTo pull the model, run: ollama pull {OLLAMA_MODEL}")
            return False

        print(f"Ollama connection OK. Using model: {OLLAMA_MODEL}")
        return True

    except requests.exceptions.ConnectionError:
        print(f"ERROR: Cannot connect to Ollama at {OLLAMA_BASE_URL}")
        print("Make sure Ollama is running with: ollama serve")
        return False
    except Exception as e:
        print(f"ERROR checking Ollama: {e}")
        return False


# Verify connection before proceeding
if not check_ollama_connection():
    raise RuntimeError("Ollama connection failed. See errors above.")

# -----------------------------------------------------------------------------
# Internal Settings
# -----------------------------------------------------------------------------

MAX_RETRIES = 5
SLEEP_BASE_SEC = 1.4
GLOBAL_MAX_CALLS = 4  # Lower for local models
_LLM_SEMAPHORE = Semaphore(GLOBAL_MAX_CALLS)

# -----------------------------------------------------------------------------
# Helper Functions
# -----------------------------------------------------------------------------


def generate_output_path(output_dir: str, prefix: str) -> str:
    """Generate output CSV path with timestamp."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prefix}_{timestamp}.csv"
    return os.path.join(output_dir, filename)


def load_prompt_text(prompt_path: str) -> str:
    """Load the bias detection prompt from file."""
    with open(prompt_path, "r", encoding="utf-8") as f:
        return f.read()


# Sentence splitting regex
_SENT_SPLIT_RE = re.compile(r"(?<=\S[.!?])\s+(?=[\"'([{]*[A-Z0-9])", re.VERBOSE)


def chunk_by_sentences(text: str, max_chars: int) -> List[str]:
    """Split text into chunks at sentence boundaries."""
    if not isinstance(text, str) or not text.strip():
        return []

    # Normalize whitespace and split into sentences
    t = re.sub(r"\s+", " ", text).strip()
    sents = [s.strip() for s in _SENT_SPLIT_RE.split(t) if s.strip()]

    if not sents:
        return []

    # Group sentences into chunks
    chunks, cur, cur_len = [], [], 0
    for s in sents:
        add_len = len(s) + (1 if cur else 0)
        if cur_len + add_len <= max_chars:
            cur.append(s)
            cur_len += add_len
        else:
            if cur:
                chunks.append(" ".join(cur).strip())
            # Handle very long sentences by hard-splitting
            if len(s) > max_chars:
                piece = s
                while len(piece) > max_chars:
                    chunks.append(piece[:max_chars])
                    piece = piece[max_chars:]
                cur, cur_len = ([piece], len(piece)) if piece else ([], 0)
            else:
                cur, cur_len = [s], len(s)
    if cur:
        chunks.append(" ".join(cur).strip())
    return chunks


def inject_chunk_into_prompt(prompt_text: str, chunk_text: str) -> str:
    """Insert note chunk into the prompt template."""
    # Try to replace the placeholder in the prompt
    placeholder = r"<<<\s*\{PASTE THE NOTE TEXT/CHUNK HERE\}\s*>>>"
    if re.search(placeholder, prompt_text):
        return re.sub(placeholder, f"<<<\n{chunk_text}\n>>>", prompt_text, count=1)

    # Fallback: replace content between <<< >>> after "INPUT NOTE CHUNK"
    block_re = re.compile(
        r"(INPUT\s+NOTE\s+CHUNK.*?<<<)([\s\S]*?)(>>>)([\s\S]*)", re.IGNORECASE
    )
    m = block_re.search(prompt_text)
    if m:
        return m.group(1) + "\n" + chunk_text + "\n" + m.group(3) + m.group(4)

    # Last resort: append at end
    return f"{prompt_text.rstrip()}\n\nINPUT NOTE CHUNK\n<<<\n{chunk_text}\n>>>"


def _sleep_backoff(attempt: int) -> None:
    """Exponential backoff with jitter for retries."""
    delay = SLEEP_BASE_SEC * (2 ** (attempt - 1)) * random.uniform(0.7, 1.3)
    time.sleep(delay)


def call_ollama(prompt: str) -> str:
    """Make a request to the Ollama API."""
    url = f"{OLLAMA_BASE_URL}/api/generate"
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": 1024,  # Max tokens to generate
        },
    }

    resp = requests.post(
        url, json=payload, timeout=300
    )  # 5 min timeout for slow models
    resp.raise_for_status()
    return resp.json().get("response", "")


def call_model_on_chunk(chunk_text: str, prompt_text: str) -> Dict[str, List[str]]:
    """Send chunk to the LLM and parse the response."""
    user_content = inject_chunk_into_prompt(prompt_text, chunk_text)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with _LLM_SEMAPHORE:
                text = call_ollama(user_content).strip()

            # Parse JSON response
            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                # Try to extract JSON object from response
                m = re.search(r"\{[\s\S]*\}", text)
                if not m:
                    # Try to find JSON array as fallback
                    m_arr = re.search(r"\[[\s\S]*\]", text)
                    if m_arr:
                        data = json.loads(m_arr.group(0))
                    else:
                        return {"possible": [], "likely": []}
                else:
                    data = json.loads(m.group(0))

            # Normalize response to expected format
            result = {"possible": [], "likely": []}
            if isinstance(data, dict):
                for key in ["possible", "likely"]:
                    if key in data and isinstance(data[key], list):
                        result[key] = [
                            x.strip() if isinstance(x, str) else str(x).strip()
                            for x in data[key]
                            if x
                        ]
            elif isinstance(data, list):
                # Backward compatibility: treat array as "likely"
                result["likely"] = [
                    x.strip() if isinstance(x, str) else str(x).strip()
                    for x in data
                    if x
                ]
            return result

        except Exception as e:
            last_err = e
            _sleep_backoff(attempt)

    print(f"[WARN] Failed chunk after {MAX_RETRIES} attempts: {last_err}")
    return {"possible": [], "likely": []}


def analyze_note_text(
    full_text: str, prompt_text: str, max_chars: int, parallel_calls: int
) -> Dict[str, List[str]]:
    """Analyze a single note and return detected bias terms."""
    if not isinstance(full_text, str) or not full_text.strip():
        return {"possible": [], "likely": []}

    chunks = chunk_by_sentences(full_text, max_chars=max_chars)
    if not chunks:
        return {"possible": [], "likely": []}

    # Process chunks in parallel (if enabled)
    if parallel_calls > 1 and len(chunks) > 1:
        results_by_idx: Dict[int, Dict[str, List[str]]] = {}
        with concurrent.futures.ThreadPoolExecutor(max_workers=parallel_calls) as ex:
            future_map = {
                ex.submit(call_model_on_chunk, ch, prompt_text): idx
                for idx, ch in enumerate(chunks)
            }
            for fut in concurrent.futures.as_completed(future_map):
                idx = future_map[fut]
                try:
                    results_by_idx[idx] = fut.result() or {"possible": [], "likely": []}
                except Exception:
                    results_by_idx[idx] = {"possible": [], "likely": []}

        # Merge results preserving order and removing duplicates
        seen_possible, seen_likely = set(), set()
        aggregated = {"possible": [], "likely": []}
        for idx in range(len(chunks)):
            chunk_result = results_by_idx.get(idx, {"possible": [], "likely": []})
            for p in chunk_result.get("possible", []):
                if p and p not in seen_possible:
                    seen_possible.add(p)
                    aggregated["possible"].append(p)
            for p in chunk_result.get("likely", []):
                if p and p not in seen_likely:
                    seen_likely.add(p)
                    aggregated["likely"].append(p)
        return aggregated

    # Sequential processing (default for local models)
    seen_possible, seen_likely = set(), set()
    aggregated = {"possible": [], "likely": []}
    for ch in chunks:
        chunk_result = call_model_on_chunk(ch, prompt_text)
        for p in chunk_result.get("possible", []):
            if p and p not in seen_possible:
                seen_possible.add(p)
                aggregated["possible"].append(p)
        for p in chunk_result.get("likely", []):
            if p and p not in seen_likely:
                seen_likely.add(p)
                aggregated["likely"].append(p)
    return aggregated


# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------

# Load the prompt
print("Loading prompt file...")
PROMPT_TEXT = load_prompt_text(PROMPT_PATH)

# Generate output path
OUTPUT_CSV = generate_output_path(OUTPUT_DIR, OUTPUT_PREFIX)

# Load input data
print(f"Loading input file: {INPUT_CSV}")
df_full = pd.read_csv(INPUT_CSV)

if "note_text" not in df_full.columns:
    raise ValueError("Input CSV must contain a 'note_text' column.")

# Select rows to process
total_rows = len(df_full)
n_to_process = (
    total_rows if N_ROWS_TO_PROCESS is None else min(N_ROWS_TO_PROCESS, total_rows)
)

if RANDOM_SELECTION and n_to_process < total_rows:
    if RANDOM_SEED is not None:
        df = df_full.sample(n=n_to_process, random_state=RANDOM_SEED).reset_index(
            drop=True
        )
    else:
        df = df_full.sample(n=n_to_process).reset_index(drop=True)
    selection_mode = (
        f"random sample (seed={RANDOM_SEED})" if RANDOM_SEED else "random sample"
    )
else:
    df = df_full.head(n_to_process).reset_index(drop=True)
    selection_mode = "first N rows" if n_to_process < total_rows else "all rows"

print(f"\n{'=' * 60}")
print("BIAS ASSESSMENT CONFIGURATION (OLLAMA)")
print(f"{'=' * 60}")
print(f"Model:             {OLLAMA_MODEL}")
print(f"Input file:        {INPUT_CSV}")
print(f"Total rows:        {total_rows}")
print(f"Rows to process:   {n_to_process} ({selection_mode})")
print(f"Output file:       {OUTPUT_CSV}")
print(f"Parallel calls:    {PARALLEL_CALLS}")
print(f"{'=' * 60}")
print("\nNOTE: Local models may be slower than cloud APIs.")
print(f"Estimated time: ~{n_to_process * 10}-{n_to_process * 30} seconds\n")

# Initialize output columns
df["Possible_Biased_Terms"] = "[]"
df["Likely_Biased_Terms"] = "[]"

# Process each note
print("Processing notes...")
start_time = time.time()

for i, (row_idx, txt) in enumerate(df["note_text"].items(), start=1):
    result = analyze_note_text(txt, PROMPT_TEXT, CHUNK_CHAR_LIMIT, PARALLEL_CALLS)
    df.at[row_idx, "Possible_Biased_Terms"] = json.dumps(
        result["possible"], ensure_ascii=False
    )
    df.at[row_idx, "Likely_Biased_Terms"] = json.dumps(
        result["likely"], ensure_ascii=False
    )
    if i % 5 == 0 or i == n_to_process:
        elapsed = time.time() - start_time
        rate = i / elapsed if elapsed > 0 else 0
        remaining = (n_to_process - i) / rate if rate > 0 else 0
        print(
            f"  Processed {i}/{n_to_process} notes... ({elapsed:.1f}s elapsed, ~{remaining:.0f}s remaining)"
        )

# Save results
df.to_csv(OUTPUT_CSV, index=False)
total_time = time.time() - start_time

print(f"\n{'=' * 60}")
print("COMPLETE")
print(f"{'=' * 60}")
print(f"Results saved to: {OUTPUT_CSV}")
print(f"Total notes processed: {len(df)}")
print(f"Total time: {total_time:.1f} seconds ({total_time / len(df):.1f}s per note)")

## Preview Results (Optional)

Run the cell below to preview the results.

In [ ]:
# =============================================================================
# RESULTS SUMMARY - Bias Detection Rates by Note Type (Human vs DAX)
# =============================================================================

print("=" * 70)
print("BIAS DETECTION RESULTS SUMMARY")
print("=" * 70)


# Helper function to check if a JSON array string is non-empty
def has_terms(json_str):
    return json_str != "[]" and json_str != ""


# Add helper columns for analysis
df["has_possible"] = df["Possible_Biased_Terms"].apply(has_terms)
df["has_likely"] = df["Likely_Biased_Terms"].apply(has_terms)
df["has_any_bias"] = df["has_possible"] | df["has_likely"]

# Overall statistics
total_notes = len(df)
notes_with_possible = df["has_possible"].sum()
notes_with_likely = df["has_likely"].sum()
notes_with_any = df["has_any_bias"].sum()

print(f"\n{'OVERALL RESULTS':^70}")
print("-" * 70)
print(f"Total notes analyzed: {total_notes}")
print("")
print(
    f"  Notes with POSSIBLE bias:  {notes_with_possible:4d}  ({100 * notes_with_possible / total_notes:5.1f}%)"
)
print(
    f"  Notes with LIKELY bias:    {notes_with_likely:4d}  ({100 * notes_with_likely / total_notes:5.1f}%)"
)
print(
    f"  Notes with ANY bias:       {notes_with_any:4d}  ({100 * notes_with_any / total_notes:5.1f}%)"
)

# Check if Dax_or_Human column exists for breakdown
if "Dax_or_Human" in df.columns:
    print(f"\n{'RESULTS BY NOTE TYPE':^70}")
    print("-" * 70)

    # Get unique note types
    note_types = df["Dax_or_Human"].unique()

    # Create summary table
    summary_data = []

    for note_type in sorted(note_types):
        subset = df[df["Dax_or_Human"] == note_type]
        n = len(subset)
        n_possible = subset["has_possible"].sum()
        n_likely = subset["has_likely"].sum()
        n_any = subset["has_any_bias"].sum()

        summary_data.append(
            {
                "Note Type": note_type,
                "Total": n,
                "Possible Bias": n_possible,
                "Possible %": f"{100 * n_possible / n:.1f}%" if n > 0 else "N/A",
                "Likely Bias": n_likely,
                "Likely %": f"{100 * n_likely / n:.1f}%" if n > 0 else "N/A",
                "Any Bias": n_any,
                "Any %": f"{100 * n_any / n:.1f}%" if n > 0 else "N/A",
            }
        )

        print(f"\n  {note_type.upper()} Notes (n={n}):")
        print(
            f"    Possible bias: {n_possible:4d}  ({100 * n_possible / n:5.1f}%)"
            if n > 0
            else "    Possible bias: N/A"
        )
        print(
            f"    Likely bias:   {n_likely:4d}  ({100 * n_likely / n:5.1f}%)"
            if n > 0
            else "    Likely bias:   N/A"
        )
        print(
            f"    Any bias:      {n_any:4d}  ({100 * n_any / n:5.1f}%)"
            if n > 0
            else "    Any bias:      N/A"
        )

    # Display as DataFrame table
    print(f"\n{'COMPARISON TABLE':^70}")
    print("-" * 70)
    summary_df = pd.DataFrame(summary_data)
    display(summary_df)

    # Calculate and display difference if both Human and Dax exist
    if "Human" in note_types and "Dax" in note_types:
        human_df = df[df["Dax_or_Human"] == "Human"]
        dax_df = df[df["Dax_or_Human"] == "Dax"]

        human_any_rate = (
            100 * human_df["has_any_bias"].sum() / len(human_df)
            if len(human_df) > 0
            else 0
        )
        dax_any_rate = (
            100 * dax_df["has_any_bias"].sum() / len(dax_df) if len(dax_df) > 0 else 0
        )

        human_likely_rate = (
            100 * human_df["has_likely"].sum() / len(human_df)
            if len(human_df) > 0
            else 0
        )
        dax_likely_rate = (
            100 * dax_df["has_likely"].sum() / len(dax_df) if len(dax_df) > 0 else 0
        )

        print(f"\n{'HUMAN vs DAX COMPARISON':^70}")
        print("-" * 70)
        print(
            f"  Any bias rate difference:    {human_any_rate - dax_any_rate:+.1f} percentage points (Human - DAX)"
        )
        print(
            f"  Likely bias rate difference: {human_likely_rate - dax_likely_rate:+.1f} percentage points (Human - DAX)"
        )

else:
    print("\n[INFO] Column 'Dax_or_Human' not found - skipping breakdown by note type.")
    print(
        "       To see Human vs DAX comparison, ensure your input CSV has a 'Dax_or_Human' column."
    )

# Clean up helper columns before final display
df_display = df.drop(
    columns=["has_possible", "has_likely", "has_any_bias"], errors="ignore"
)

print(f"\n{'SAMPLE FLAGGED NOTES':^70}")
print("-" * 70)

# Show notes that have at least one flagged term
flagged = df[df["has_any_bias"]]
print(f"Notes with flagged terms: {len(flagged)} / {len(df)}")

if len(flagged) > 0:
    print("\nFirst few flagged notes:")
    display_cols = ["Likely_Biased_Terms", "Possible_Biased_Terms"]
    if "Dax_or_Human" in df.columns:
        display_cols = ["Dax_or_Human"] + display_cols
    display(flagged[display_cols].head(10))